# Tuned Linear SVM on TF-IDF

This notebook tunes `LinearSVC` and evaluates the best model on a held-out test set.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import LinearSVC

In [ ]:
TFIDF_DIR = Path("data/processed/tfidf")
X_PATH = TFIDF_DIR / "X_tfidf.npz"
Y_PATH = TFIDF_DIR / "y_labels.csv"

if not X_PATH.exists() or not Y_PATH.exists():
    raise FileNotFoundError(
        "Missing TF-IDF artifacts. Run these first:\n"
        "1) python preprocess_data.py\n"
        "2) python vectorize_tfidf.py"
    )

In [ ]:
X = load_npz(X_PATH)
y = pd.read_csv(Y_PATH)["class"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X shape: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Class distribution:")
print(pd.Series(y).value_counts().sort_index())

In [ ]:
baseline_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    loss="squared_hinge",
    random_state=42,
)
baseline_svm.fit(X_train, y_train)
baseline_pred = baseline_svm.predict(X_test)

print("Baseline LinearSVC metrics:")
print(f"accuracy: {accuracy_score(y_test, baseline_pred):.4f}")
print(f"f1_macro: {f1_score(y_test, baseline_pred, average='macro', zero_division=0):.4f}")
print(f"class0_f1: {classification_report(y_test, baseline_pred, output_dict=True, zero_division=0)['0']['f1-score']:.4f}")

In [ ]:
param_grid = {
    "C": [0.1, 0.5, 1.0, 2.0, 5.0],
    "class_weight": ["balanced", {0: 3, 1: 1, 2: 1.5}, {0: 4, 1: 1, 2: 1.5}],
    "loss": ["hinge", "squared_hinge"],
}

grid = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=1,
    verbose=1,
)

grid.fit(X_train, y_train)
best_svm = grid.best_estimator_

print("Best params:", grid.best_params_)
print(f"Best CV f1_macro: {grid.best_score_:.4f}")

In [ ]:
tuned_pred = best_svm.predict(X_test)

metrics_df = pd.DataFrame([
    {
        "model": "LinearSVC (baseline)",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "precision_macro": precision_score(y_test, baseline_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, baseline_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, baseline_pred, average="weighted", zero_division=0),
        "class0_f1": classification_report(y_test, baseline_pred, output_dict=True, zero_division=0)["0"]["f1-score"],
        "class0_recall": classification_report(y_test, baseline_pred, output_dict=True, zero_division=0)["0"]["recall"],
    },
    {
        "model": "LinearSVC (tuned)",
        "accuracy": accuracy_score(y_test, tuned_pred),
        "precision_macro": precision_score(y_test, tuned_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, tuned_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, tuned_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, tuned_pred, average="weighted", zero_division=0),
        "class0_f1": classification_report(y_test, tuned_pred, output_dict=True, zero_division=0)["0"]["f1-score"],
        "class0_recall": classification_report(y_test, tuned_pred, output_dict=True, zero_division=0)["0"]["recall"],
    },
]).set_index("model")

metrics_df.round(4)

In [ ]:
print("Tuned LinearSVC classification report:\n")
print(classification_report(y_test, tuned_pred, digits=4, zero_division=0))

labels = np.sort(np.unique(y))
cm = confusion_matrix(y_test, tuned_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

## Optional

Append your tuned SVM metrics to `model_metrics_summary.txt` if you want one consolidated report.